# [SQL 실습 #10] MySQL GROUP BY 기초
## 학년별, 학과별로 묶어서 집계하기

> **학생용 실습 노트북 - TODO + 자동점검형**

이번 실습은 **SQL-10.pdf**의 흐름에 맞추어 구성했습니다.

지난 시간에는 `COUNT`, `SUM`, `AVG`, `MAX`, `MIN`을 이용해 전체 데이터를 집계했습니다.  
이번 시간에는 `GROUP BY`를 이용해 데이터를 **학년별, 학과별로 묶어서** 각각 집계합니다.

### 오늘의 학습 목표
1. `GROUP BY`의 의미 이해하기
2. 학년별/학과별 학생 수 구하기
3. 학년별/학과별 평균 구하기
4. 학년별 최고·최저 점수 구하기
5. 여러 집계 함수를 한 번에 사용하기
6. `GROUP BY + ORDER BY`
7. `WHERE + GROUP BY`
8. `COUNT(*)`와 `COUNT(score)` 비교하기
9. `HAVING` 맛보기

> **GROUP BY는 같은 값을 가진 데이터끼리 묶어서 집계할 때 사용하는 SQL 문법입니다.**


# 1. MySQL 실습 환경 준비

아래 셀은 Google Colab에서 MySQL 서버를 설치하고 실행합니다.

> 수정하지 말고 실행하세요.


In [ ]:
!sudo apt-get -qq update
!sudo DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mysql-server > /dev/null
!sudo service mysql start
print("✅ MySQL 서버 준비 완료")

# 2. 실습 데이터 준비

이번 실습은 12명의 학생 데이터를 사용합니다.

| id | name | grade | class_name | score |
|---:|---|---:|---|---:|
|1|김부장|3|컴퓨터과|85|
|2|훈이|1|영훈반|90|
|3|라즈베리|2|임베디드반|78|
|4|리눅스|3|서버반|88|
|5|파이썬|1|프로그래밍반|95|
|6|김코딩|1|컴퓨터과|82|
|7|데이터왕|2|데이터베이스반|91|
|8|홍길동|2|컴퓨터과|76|
|9|이몽룡|3|데이터베이스반|89|
|10|정렬이|1|SQL반|NULL|
|11|제한이|2|SQL반|84|
|12|그룹이|3|컴퓨터과|92|

`정렬이` 학생의 `score`는 `NULL`입니다.


In [ ]:
%%bash
sudo mysql <<'SQL'
CREATE DATABASE IF NOT EXISTS school
DEFAULT CHARACTER SET utf8mb4
DEFAULT COLLATE utf8mb4_unicode_ci;

USE school;
DROP TABLE IF EXISTS students;

CREATE TABLE students (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(50) NOT NULL,
    grade INT,
    class_name VARCHAR(50),
    score INT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO students (name, grade, class_name, score) VALUES
('김부장',3,'컴퓨터과',85),
('훈이',1,'영훈반',90),
('라즈베리',2,'임베디드반',78),
('리눅스',3,'서버반',88),
('파이썬',1,'프로그래밍반',95),
('김코딩',1,'컴퓨터과',82),
('데이터왕',2,'데이터베이스반',91),
('홍길동',2,'컴퓨터과',76),
('이몽룡',3,'데이터베이스반',89),
('정렬이',1,'SQL반',NULL),
('제한이',2,'SQL반',84),
('그룹이',3,'컴퓨터과',92);

SELECT id,name,grade,class_name,score
FROM students
ORDER BY id;
SQL

# 3. SQL 실행 / 자동점검 도우미

- `TODO`가 남아 있으면 실행하지 않습니다.
- 학생 실습 셀은 조회 전용입니다.
- 자동점검은 결과값과 순서를 비교합니다.

> 아래 셀은 수정하지 마세요.


In [ ]:
import subprocess, textwrap, re
TODO_STATUS = {}

def _mysql(sql, no_header=False):
    cmd = ["sudo","mysql","--batch","--raw"]
    if no_header:
        cmd.append("-N")
    p = subprocess.run(cmd, input=textwrap.dedent(sql).strip(),
                       text=True, capture_output=True)
    return p.returncode == 0, p.stdout.strip(), p.stderr.strip()

def _clean(sql):
    return "\n".join(
        x for x in textwrap.dedent(sql).splitlines()
        if x.strip() and not x.lstrip().startswith("--")
    ).strip()

def run_sql(sql):
    real = _clean(sql)
    if "TODO" in real:
        print("⚠️ TODO가 남아 있습니다.")
        return False
    if re.search(r"\b(INSERT|UPDATE|DELETE|DROP|TRUNCATE|ALTER|CREATE)\b", real, re.I):
        print("❌ TODO 셀은 조회 전용입니다.")
        return False
    ok,out,err = _mysql(sql)
    if out: print(out)
    if not ok:
        print("❌ MySQL 오류")
        print(err)
        return False
    print("✅ 조회 완료")
    return True

def _result(sql):
    ok,out,err = _mysql(sql, True)
    if not ok:
        raise RuntimeError(err)
    return out.splitlines() if out else []

def check_todo(n, student_sql, answer_sql):
    if "TODO" in _clean(student_sql):
        TODO_STATUS[n] = False
        print(f"🔎 TODO {n}: ❌ 먼저 TODO를 완성하세요.")
        return False
    try:
        actual = _result(student_sql)
        expected = _result(answer_sql)
        passed = actual == expected
    except Exception as e:
        TODO_STATUS[n] = False
        print(f"🔎 TODO {n}: ❌ SQL 오류 - {e}")
        return False
    TODO_STATUS[n] = passed
    print(f"🔎 TODO {n}: " + ("✅ 통과" if passed else "❌ 다시 확인"))
    if not passed:
        print("예상:", expected)
        print("현재:", actual)
    return passed

def show_progress(total):
    print("===== TODO 진행 현황 =====")
    passed = 0
    for i in range(1,total+1):
        s = TODO_STATUS.get(i)
        mark = "✅" if s is True else ("❌" if s is False else "⬜")
        passed += (s is True)
        print(f"{mark} TODO {i}")
    print(f"\n진행률: {passed}/{total}")


# 4. GROUP BY란?

전체 학생 수는 다음처럼 구합니다.

```sql
SELECT COUNT(*)
FROM students;
```

하지만 **학년별**, **학과별**로 각각 계산하려면 `GROUP BY`가 필요합니다.

```sql
SELECT grade, COUNT(*)
FROM students
GROUP BY grade;
```

> 학생 전체를 한 번에 세는 것이 아니라, **1학년 바구니 / 2학년 바구니 / 3학년 바구니로 나눈 뒤 각각 세는 것**입니다.


# 5. 학년별 학생 수 구하기

## TODO 1

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 학생수
FROM students
GROUP BY TODO
ORDER BY grade ASC;
'''
run_sql(sql)

In [ ]:
check_todo(
    1,
    sql,
    'SELECT grade, COUNT(*) AS 학생수 FROM students GROUP BY grade ORDER BY grade ASC;'
)

# 6. 학과별 학생 수 구하기

## TODO 2

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 학생수
FROM students
GROUP BY TODO
ORDER BY class_name ASC;
'''
run_sql(sql)

In [ ]:
check_todo(
    2,
    sql,
    'SELECT class_name, COUNT(*) AS 학생수 FROM students GROUP BY class_name ORDER BY class_name ASC;'
)

# 7. 학년별 평균 점수 구하기

## TODO 3

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 평균점수
FROM students
GROUP BY TODO
ORDER BY grade ASC;
'''
run_sql(sql)

In [ ]:
check_todo(
    3,
    sql,
    'SELECT grade, ROUND(AVG(score),1) AS 평균점수 FROM students GROUP BY grade ORDER BY grade ASC;'
)

# 8. 학년별 최고점수와 최저점수 구하기

## TODO 4

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 최고점수,
    TODO AS 최저점수
FROM students
GROUP BY TODO
ORDER BY grade ASC;
'''
run_sql(sql)

In [ ]:
check_todo(
    4,
    sql,
    'SELECT grade, MAX(score) AS 최고점수, MIN(score) AS 최저점수 FROM students GROUP BY grade ORDER BY grade ASC;'
)

# 9. 학년별 여러 집계 함수 한 번에 사용하기

## TODO 5

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 학생수,
    TODO AS 점수입력수,
    TODO AS 점수합계,
    TODO AS 평균점수,
    TODO AS 최고점수,
    TODO AS 최저점수
FROM students
GROUP BY TODO
ORDER BY grade ASC;
'''
run_sql(sql)

In [ ]:
check_todo(
    5,
    sql,
    'SELECT grade, COUNT(*) AS 학생수, COUNT(score) AS 점수입력수,\nSUM(score) AS 점수합계, ROUND(AVG(score),1) AS 평균점수,\nMAX(score) AS 최고점수, MIN(score) AS 최저점수\nFROM students GROUP BY grade ORDER BY grade ASC;'
)

# 10. 학과별 평균 점수 구하기

## TODO 6

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 평균점수
FROM students
GROUP BY TODO
ORDER BY class_name ASC;
'''
run_sql(sql)

In [ ]:
check_todo(
    6,
    sql,
    'SELECT class_name, ROUND(AVG(score),1) AS 평균점수 FROM students GROUP BY class_name ORDER BY class_name ASC;'
)

# 11. 학년별 평균 점수를 높은 순서로 정렬하기

## TODO 7

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 평균점수
FROM students
GROUP BY TODO
ORDER BY TODO DESC;
'''
run_sql(sql)

In [ ]:
check_todo(
    7,
    sql,
    'SELECT grade, ROUND(AVG(score),1) AS 평균점수 FROM students GROUP BY grade ORDER BY 평균점수 DESC;'
)

# 12. 학과별 학생 수를 많은 순서로 정렬하기

## TODO 8

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 학생수
FROM students
GROUP BY TODO
ORDER BY TODO DESC, class_name ASC;
'''
run_sql(sql)

In [ ]:
check_todo(
    8,
    sql,
    'SELECT class_name, COUNT(*) AS 학생수 FROM students GROUP BY class_name ORDER BY 학생수 DESC, class_name ASC;'
)

# 13. 80점 이상 학생만 대상으로 학년별 학생 수 구하기

## TODO 9

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 80점이상학생수
FROM students
WHERE TODO
GROUP BY TODO
ORDER BY grade ASC;
'''
run_sql(sql)

In [ ]:
check_todo(
    9,
    sql,
    'SELECT grade, COUNT(*) AS 80점이상학생수 FROM students WHERE score >= 80 GROUP BY grade ORDER BY grade ASC;'
)

# 14. 80점 이상 학생만 대상으로 학과별 평균 구하기

## TODO 10

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 평균점수
FROM students
WHERE TODO
GROUP BY TODO
ORDER BY TODO DESC;
'''
run_sql(sql)

In [ ]:
check_todo(
    10,
    sql,
    'SELECT class_name, ROUND(AVG(score),1) AS 평균점수 FROM students WHERE score >= 80 GROUP BY class_name ORDER BY 평균점수 DESC;'
)

# 15. 학과별 전체학생수와 점수입력학생수 비교하기

## TODO 11

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 전체학생수,
    TODO AS 점수입력학생수
FROM students
GROUP BY TODO
ORDER BY class_name ASC;
'''
run_sql(sql)

In [ ]:
check_todo(
    11,
    sql,
    'SELECT class_name, COUNT(*) AS 전체학생수, COUNT(score) AS 점수입력학생수 FROM students GROUP BY class_name ORDER BY class_name ASC;'
)

# 16. 평균점수가 85점 이상인 학과만 보기 - HAVING 맛보기

## TODO 12

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 평균점수
FROM students
GROUP BY TODO
HAVING TODO
ORDER BY TODO DESC;
'''
run_sql(sql)

In [ ]:
check_todo(
    12,
    sql,
    'SELECT class_name, ROUND(AVG(score),1) AS 평균점수 FROM students GROUP BY class_name HAVING AVG(score) >= 85 ORDER BY 평균점수 DESC;'
)

# 17. GROUP BY 기본 문법 정리

```sql
SELECT 그룹기준컬럼, 집계함수
FROM 테이블명
GROUP BY 그룹기준컬럼;
```

예:

```sql
SELECT grade, COUNT(*)
FROM students
GROUP BY grade;
```

```sql
SELECT class_name, ROUND(AVG(score),1)
FROM students
GROUP BY class_name;
```

### 중요한 원칙
1. `GROUP BY`의 기준 컬럼을 `SELECT`에 표시합니다.
2. 나머지 값은 `COUNT`, `SUM`, `AVG`, `MAX`, `MIN` 같은 집계 함수로 계산합니다.


# 18. WHERE와 GROUP BY의 실행 흐름

```text
FROM
 ↓
WHERE       ← 먼저 필요한 행 선택
 ↓
GROUP BY    ← 선택된 행을 그룹으로 묶기
 ↓
SELECT      ← 집계 함수 계산
 ↓
HAVING      ← 그룹 결과에 조건 적용
 ↓
ORDER BY    ← 최종 결과 정렬
```

- `WHERE` : 그룹으로 묶기 **전**
- `HAVING` : 그룹으로 묶은 **후**


# 19. 자주 발생하는 실수

### 1) GROUP BY 없이 학과별 평균을 구하려는 경우

```sql
SELECT class_name, AVG(score)
FROM students;
```

학과별 평균이라면:

```sql
SELECT class_name, AVG(score)
FROM students
GROUP BY class_name;
```

### 2) GROUP BY 위치를 잘못 쓰는 경우

기본 순서:

```text
SELECT
FROM
WHERE
GROUP BY
HAVING
ORDER BY
```

### 3) SELECT에 아무 컬럼이나 같이 쓰는 경우

```sql
SELECT name, grade, COUNT(*)
FROM students
GROUP BY grade;
```

`grade` 하나의 그룹 안에 여러 이름이 있기 때문에 어떤 이름을 보여줘야 하는지 명확하지 않습니다.

### 4) WHERE와 HAVING을 헷갈리는 경우

- `WHERE` → 개별 행에 조건
- `HAVING` → 그룹 집계 결과에 조건


# 20. 종합 실습

### 문제 A
학년별 점수 합계를 구하세요.

### 문제 B
학과별 최고 점수와 최저 점수를 구하세요.

### 문제 C
2학년 학생만 대상으로 학과별 학생 수를 구하세요.

### 문제 D
학과별 평균 점수를 구한 뒤 높은 순서로 정렬하세요.

### 문제 E ★
학생이 2명 이상인 학과만 출력하세요.

<details>
<summary>정답 보기</summary>

```sql
-- A
SELECT grade, SUM(score) AS 점수합계
FROM students
GROUP BY grade;

-- B
SELECT class_name,
       MAX(score) AS 최고점수,
       MIN(score) AS 최저점수
FROM students
GROUP BY class_name;

-- C
SELECT class_name, COUNT(*) AS 학생수
FROM students
WHERE grade = 2
GROUP BY class_name;

-- D
SELECT class_name,
       ROUND(AVG(score),1) AS 평균점수
FROM students
GROUP BY class_name
ORDER BY 평균점수 DESC;

-- E
SELECT class_name, COUNT(*) AS 학생수
FROM students
GROUP BY class_name
HAVING COUNT(*) >= 2;
```

</details>


# 21. 오늘 배운 내용 정리

| 문법 | 역할 |
|---|---|
| `GROUP BY` | 같은 값을 가진 데이터를 그룹으로 묶기 |
| `COUNT()` | 개수 |
| `SUM()` | 합계 |
| `AVG()` | 평균 |
| `MAX()` | 최댓값 |
| `MIN()` | 최솟값 |
| `WHERE` | 그룹화 전에 행 조건 적용 |
| `HAVING` | 그룹화 후 집계 결과에 조건 적용 |
| `ORDER BY` | 결과 정렬 |

> **GROUP BY는 데이터를 바구니별로 나누고, 각 바구니의 개수·합계·평균을 구하는 문법입니다.**


# 22. 전체 TODO 진행 현황

In [ ]:
show_progress(12)

# 23. 자기 점검

- [ ] `GROUP BY`의 의미를 설명할 수 있다.
- [ ] 학년별 학생 수를 구할 수 있다.
- [ ] 학과별 학생 수를 구할 수 있다.
- [ ] 학년별 평균을 구할 수 있다.
- [ ] 학년별 최고/최저 점수를 구할 수 있다.
- [ ] 여러 집계 함수를 함께 사용할 수 있다.
- [ ] 학과별 평균을 구할 수 있다.
- [ ] `GROUP BY + ORDER BY`를 사용할 수 있다.
- [ ] `WHERE + GROUP BY`를 사용할 수 있다.
- [ ] `COUNT(*)`와 `COUNT(score)`의 차이를 설명할 수 있다.
- [ ] `WHERE`와 `HAVING`의 차이를 대략 설명할 수 있다.


# 다음 시간 예고

## [SQL 실습 #11] MySQL HAVING 기초
### GROUP BY 결과에 조건 걸기

다음 시간에는 `GROUP BY`로 만든 결과에 조건을 거는 `HAVING`을 자세히 배웁니다.
